# Pricing Simulator — Statistical inference (spec section 9)

**Part 4.** Uses the same helpers as `GET /api/runs/{run_id}/experiment-inference` (`app/services/stats/inference.py`): Wilson score intervals for binomial conversion and a pooled two-proportion z-test.

Prerequisites: `pip install -e ".[dev]"` from the repo root. No database required for the **pure math** cells below. To mirror the API on a completed run, use PostgreSQL and the optional cell at the end.

See also: `docs/spec-mapping.md`, `POST /api/runs/batch`, and `scripts/run_batch_seeds.py` for multi-seed runs.

**Bayesian comparison:** For Beta–binomial posteriors, credible intervals, and P(variant better) side by side with this frequentist story, continue to [`05_bayesian_experiment_inference.ipynb`](05_bayesian_experiment_inference.ipynb).


In [ ]:
from app.services.stats.inference import (
    ExperimentArmRollup,
    build_experiment_inference,
    two_proportion_z_test_p_value,
    wilson_interval,
)

# Toy arm totals: 120 orders / 2000 customer-days vs 150 / 2000
ctrl = ExperimentArmRollup(customer_days=2000, orders=120, net_revenue=0.0, contribution_margin=0.0)
var = ExperimentArmRollup(customer_days=2000, orders=150, net_revenue=0.0, contribution_margin=0.0)
out = build_experiment_inference(run_id="toy", control=ctrl, variant=var)
print("Wilson 95% CI control:", (out.control.conversion_rate_wilson_low, out.control.conversion_rate_wilson_high))
print("Wilson 95% CI variant:", (out.variant.conversion_rate_wilson_low, out.variant.conversion_rate_wilson_high))
print("z-stat, p (two-sided):", out.two_proportion_z_statistic, out.two_proportion_p_value_two_sided)
print("Lift (absolute, relative):", out.conversion_lift_absolute, out.conversion_lift_relative)
print("Bayesian P(variant > control):", out.bayesian.comparison.prob_variant_superior)
print("Bayesian 95% cred. control:", (out.bayesian.control.conversion_rate_credible_low, out.bayesian.control.conversion_rate_credible_high))


## Wilson interval building blocks

`wilson_interval(successes, n)` returns the Agresti–Coull-style Wilson score bounds for a binomial proportion at z=1.96.


In [ ]:
print(wilson_interval(45, 500))
z, p = two_proportion_z_test_p_value(45, 500, 60, 500)
print("z, p:", z, p)
